# Phase 2.4 — Exploratory Data Analysis: Training Soil Data (`2015–2025`)

**Project:** G2F Maize Yield Prediction 2026  
**Primary Dataset:** `3_Training_Soil_Data_2015_2025.csv`  
**Read-Only Cross-References (Keys/Target Only):** `1_Training_Trait_Data_2014_2025.csv` (`Env`, `Year`, `Yield_Mg_ha`) and `2_Training_Meta_Data_2014_2025.csv` (`Env`, `Year`)

## Why Soil Data Matters for Maize Yield Prediction
Soil acts as the physical, chemical, and biological reservoir supporting every maize plant:
- **Physical properties** (e.g., `% Sand`, `% Silt`, `% Clay`, USDA `Texture`) govern water-holding capacity, drainage, aeration, and root penetration.
- **Chemical acidity & salinity** (e.g., `1:1 Soil pH`, buffer pH, soluble salts) determine whether soil nutrients are chemically bioavailable to roots or locked up/toxic.
- **Fertility & exchange capacity** (e.g., `Organic Matter LOI %`, `CEC/Sum of Cations me/100g`, nitrate-N, potassium, phosphorus, calcium, magnesium, sulfur) reflect baseline nutrient supply and buffering capacity.

In a Genotype × Environment (G × E) ML system, soil properties help explain *why* two fields with similar weather behave very differently — **provided** we verify coverage across years/environments, laboratory consistency, sampling timing (pre-season vs. post-season), and availability in the 2026 testing set.

## Scope & Guardrails
- Strictly read-only EDA: no raw files are modified, cleaned, imputed, normalized, aggregated, or merged into persisted tables.
- No production feature engineering, model training, or inspection of weather/genotype/EC datasets.
- All statistics and chart titles are computed dynamically at runtime.
- Per `DEC-007`, committed versions of this notebook have all outputs and execution counts stripped; full data-derived reports (`docs/SOIL_DATA_EDA.md`) and visualizations (`reports/figures/local/`) remain strictly local and gitignored.

## 1. Reproducible Data Loading

We resolve dataset paths relative to the repository root, inspect the raw CSV header and first 5 sample rows using Pandas' C engine (`nrows=5`), and then load the complete soil table with `low_memory=False` while recording dimensions, load time, and deep memory usage.

In [ ]:
import re
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid", context="notebook")


def find_project_root(start: Path) -> Path:
    """Walk upward from start until the repository root (containing README.md and data/) is found."""
    for candidate in [start, *start.parents]:
        if (candidate / "README.md").exists() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate project root containing README.md and data/.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
RAW_TRAIN_DIR = PROJECT_ROOT / "data/raw/training/Training_Data-20261123T123657Z-1-001/Training_Data"
SOIL_PATH = RAW_TRAIN_DIR / "3_Training_Soil_Data_2015_2025.csv"
TRAIT_PATH = RAW_TRAIN_DIR / "1_Training_Trait_Data_2014_2025.csv"
META_PATH = RAW_TRAIN_DIR / "2_Training_Meta_Data_2014_2025.csv"
LOCAL_FIGURES_DIR = PROJECT_ROOT / "reports/figures/local"
LOCAL_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

assert SOIL_PATH.is_file(), f"Soil dataset not found: {SOIL_PATH.relative_to(PROJECT_ROOT)}"
assert TRAIT_PATH.is_file(), f"Trait dataset not found: {TRAIT_PATH.relative_to(PROJECT_ROOT)}"
assert META_PATH.is_file(), f"Metadata dataset not found: {META_PATH.relative_to(PROJECT_ROOT)}"

print(f"Soil relative path   : {SOIL_PATH.relative_to(PROJECT_ROOT).as_posix()}")
print(f"File size on disk    : {SOIL_PATH.stat().st_size / 1024:.2f} KB ({SOIL_PATH.stat().st_size:,} bytes)")
print(f"Local figures folder : {LOCAL_FIGURES_DIR.relative_to(PROJECT_ROOT).as_posix()}")

In [ ]:
# Step 1a: Inspect raw CSV header and first 5 sample records before full load
preview_soil = pd.read_csv(SOIL_PATH, engine="c", nrows=5)
print(f"Preview shape: {preview_soil.shape}")
print(f"Column count : {len(preview_soil.columns)}")
display(preview_soil)

In [ ]:
# Step 1b: Load complete soil dataset preserving original column names and values
t0 = time.perf_counter()
soil_df = pd.read_csv(SOIL_PATH, engine="c", low_memory=False)
load_seconds = time.perf_counter() - t0

mem_bytes_deep = int(soil_df.memory_usage(deep=True).sum())
mem_kb_deep = mem_bytes_deep / 1024
mem_mb_deep = mem_bytes_deep / (1024 ** 2)

print(f"Loaded shape          : {soil_df.shape[0]:,} rows x {soil_df.shape[1]} columns")
print(f"Load time (C engine)  : {load_seconds:.4f} seconds")
print(f"Memory usage (deep)   : {mem_kb_deep:.2f} KB ({mem_mb_deep:.4f} MB | {mem_bytes_deep:,} bytes)")

## 2. Schema, Missingness & Scientific Interpretation

### Documentation Boundary
No separate soil data dictionary exists inside `data/raw/training/`. However, many column headers in `3_Training_Soil_Data_2015_2025.csv` explicitly encode the **laboratory method and/or measurement unit** (e.g., `mmho/cm`, `LOI %`, `ppm N`, `me/100g`, `Mehlich P-III ppm P`). Below we strictly distinguish:
- **Verified (from column header):** Method and/or unit explicitly printed in the CSV header.
- **Inferred:** Agronomic meaning inferred from standard U.S. agricultural soil testing laboratory conventions.
- **Unknown / Requires confirmation:** Ambiguous shorthand or undocumented unit (e.g., `E Depth` unit — inches vs. cm — and `Texture No` numeric coding scheme).

### Beginner-Friendly Guide to Soil Measurements Present in This Dataset
- **`1:1 Soil pH`:** Soil acidity/alkalinity measured in a 1:1 soil-to-water slurry (unitless scale `0–14`). Maize generally thrives near pH `6.0–7.0`; extreme acidity (`< 5.5`) can increase aluminum toxicity and reduce phosphorus availability, while high alkalinity (`> 7.5`) can limit micronutrient availability.
- **`WDRF Buffer pH` & `BpH`:** Buffer pH measurements (Woodruff / Sikora-style buffer tests) used by labs to estimate **reserve soil acidity** and lime requirement when soil pH is acidic. Different labs often report under different column names.
- **`1:1 S Salts mmho/cm`:** Soluble salts / electrical conductivity (`mmho/cm`, equivalent to `dS/m`) in a 1:1 soil-water extract. High salinity creates osmotic stress, making it harder for maize roots to absorb water.
- **`Organic Matter LOI %`:** Soil organic matter percentage measured by **Loss on Ignition (LOI)**. Higher organic matter improves water retention, soil structure, microbial activity, and slow-release nitrogen supply.
- **`Nitrate-N ppm N` & `lbs N/A`:** Plant-available soil nitrate nitrogen concentration (`ppm N`) and estimated nitrogen supply per acre (`lbs N/A`). Unlike texture or organic matter, nitrate is **highly mobile** (leaches with rain and fluctuates rapidly with fertilizer timing and crop uptake).
- **`Mehlich P-III ppm P`:** Extractable soil phosphorus (`ppm P`) measured via the Mehlich-3 extraction method — essential for early root development, energy transfer (ATP), and grain fill.
- **Exchangeable Cations & Sulfate (`Potassium ppm K`, `Calcium ppm Ca`, `Magnesium ppm Mg`, `Sodium ppm Na`, `Sulfate-S ppm S`):** Key plant macronutrients (`K`, `Ca`, `Mg`, `S`) plus sodium (`Na`, excess of which can degrade soil structure).
- **`CEC/Sum of Cations me/100g`:** Cation Exchange Capacity in milliequivalents per 100 g of soil (`me/100g`, equivalent to `cmol(+)/kg`), reflecting the soil's capacity to hold positively charged nutrients (`K+`, `Ca2+`, `Mg2+`). Clay- and organic-matter-rich soils have higher CEC than sandy soils.
- **Cation Saturation Percentages (`%H Sat`, `%K Sat`, `%Ca Sat`, `%Mg Sat`, `%Na Sat`):** Percentage of the soil's exchange sites occupied by each cation (acidic `H+` vs. basic cations `K+`, `Ca2+`, `Mg2+`, `Na+`); physically, the five percentages should sum near `100%`.
- **Soil Texture (`% Sand`, `% Silt`, `% Clay`, `Texture`, `Texture No`):** Particle-size composition (`% Sand + % Silt + % Clay` should sum to `100%`) and corresponding USDA textural class (e.g., `Clay Loam`, `Sandy Loam`, `Silt Loam`), strongly controlling water availability and drought resilience.
- **Micronutrients (`Zinc ppm Zn`, `Iron ppm Fe`, `Manganese ppm Mn`, `Copper ppm Cu`, `Boron ppm B`):** Trace metals/elements essential in tiny quantities; inspect their missingness before considering them as features!

In [ ]:
# Complete 36-column schema, cardinality, and missingness summary
schema_df = pd.DataFrame({
    "column": soil_df.columns,
    "inferred_dtype": soil_df.dtypes.astype(str).values,
    "non_missing": soil_df.notna().sum().values,
    "missing_count": soil_df.isna().sum().values,
    "missing_pct": (soil_df.isna().mean() * 100).round(2).values,
    "nunique_non_null": [int(soil_df[c].nunique(dropna=True)) for c in soil_df.columns],
    "memory_kb_deep": [round(soil_df[c].memory_usage(deep=True) / 1024, 2) for c in soil_df.columns],
})

print(f"Total rows: {len(soil_df):,} | Total columns: {soil_df.shape[1]}")
print("\n=== SOIL DATASET SCHEMA & MISSINGNESS ===")
print(schema_df.to_string(index=False))

## 3. Data Quality Analysis

### Why we audit raw tokens separately from Pandas default parsing
By default, `pd.read_csv()` converts strings like `"NA"`, `"None"`, or `"NaN"` into `NaN`, while leaving strings like `"< 0.01"` (laboratory below-detection-limit notation!) as text — which forces an entire numerical column into `str`/`object` dtype! Here we inspect:
1. Exact duplicate rows and missing identifiers.
2. Non-numeric tokens hiding inside numerical-looking columns.
3. Explicit `+inf` / `-inf` finite checks.
4. Raw CSV missing-value tokens (`keep_default_na=False`) separately from Pandas defaults.
5. Categorical value consistency (`LabID`, `Texture`, `Comments`) and date-parsing validity (`Date Received`, `Date Reported`).
6. Physical/scientific plausibility checks (texture sums, cation saturation sums, pH bounds, negative concentrations) — strictly **diagnostic**, without replacing or removing any values, and **without** treating legitimate zeros (such as `0%` `%H Sat` in alkaline soils) as missing!

In [ ]:
# 1) Exact duplicates & missing identifiers
exact_dups = int(soil_df.duplicated().sum())
id_missing = {c: int(soil_df[c].isna().sum()) for c in ["Year", "Env", "LabID"]}

print("=== 1) DUPLICATES & IDENTIFIER COMPLETENESS ===")
print(f"Exact duplicate rows across all columns: {exact_dups:,}")
print(f"Missing identifier counts              : {id_missing}")

# 2) Numeric parsing check on string columns that look numeric (e.g., %H Sat, %Na Sat)
print("\n=== 2) NUMERIC PARSING PROBLEMS IN STRING COLUMNS ===")
non_numeric_issues = {}
expected_text_cols = {"Env", "LabID", "Date Received", "Date Reported", "Texture", "Comments"}
for col in soil_df.select_dtypes(include=["object", "string"]).columns:
    if col in expected_text_cols:
        continue
    raw_non_null = soil_df[col].dropna().astype(str).str.strip()
    coerced = pd.to_numeric(raw_non_null, errors="coerce")
    bad = raw_non_null[coerced.isna()]
    non_numeric_issues[col] = {
        "non_null": int(len(raw_non_null)),
        "numeric_valid": int(coerced.notna().sum()),
        "non_numeric_count": int(len(bad)),
        "non_numeric_tokens": bad.value_counts().to_dict(),
    }
    print(f"  {col}: {non_numeric_issues[col]}")

# 3) Raw CSV NA-like tokens (keep_default_na=False) vs Pandas default parsing
print("\n=== 3) RAW CSV MISSING-VALUE & SENTINEL TOKEN AUDIT (keep_default_na=False) ===")
candidate_sentinels = {
    "", " ", "na", "n/a", "nan", "null", "none", ".", "-", "--", "?", "missing",
    "unknown", "nd", "n.d.", "bdl", "<0.01", "< 0.01", "-999", "-99", "999", "9999",
}
raw_str_df = pd.read_csv(SOIL_PATH, engine="c", dtype=str, keep_default_na=False, low_memory=False)
pandas_default_na = {
    "", "#N/A", "#N/A N/A", "#NA", "-1.#IND", "-1.#QNAN", "-NaN", "-nan",
    "1.#IND", "1.#QNAN", "<NA>", "N/A", "NA", "NULL", "NaN", "None", "n/a", "nan", "null"
}
any_non_empty_token = False
for col in raw_str_df.columns:
    s = raw_str_df[col]
    hits = s[s.isin(pandas_default_na) | s.str.strip().str.lower().isin(candidate_sentinels)]
    vc = hits.value_counts().to_dict()
    non_empty = {k: v for k, v in vc.items() if k != ""}
    if non_empty:
        any_non_empty_token = True
        print(f"  {col}: empty='' ({vc.get('', 0)}), special_tokens={non_empty}")
if not any_non_empty_token:
    print("  All raw missing cells are empty strings '' (no explicit NA/sentinel text tokens).")
del raw_str_df

# 4) Positive / negative infinity & sentinel numeric code audit across all numeric + numeric-string cols
print("\n=== 4) INFINITY & SENTINEL NUMERIC CODE AUDIT ===")
num_sentinel_codes = [-9999, -999, -99, -1, 999, 9999]
numeric_candidate_cols = [c for c in soil_df.columns if c not in expected_text_cols]
inf_audit_rows = []
for col in numeric_candidate_cols:
    coerced = pd.to_numeric(soil_df[col], errors="coerce")
    pos_inf = int(np.isposinf(coerced).sum())
    neg_inf = int(np.isneginf(coerced).sum())
    fin = coerced[np.isfinite(coerced)]
    code_hits = {code_v: int((fin == code_v).sum()) for code_v in num_sentinel_codes if (fin == code_v).any()}
    if pos_inf or neg_inf or code_hits or int((fin < 0).sum()) > 0:
        inf_audit_rows.append({
            "column": col,
            "pos_inf": pos_inf,
            "neg_inf": neg_inf,
            "n_negative": int((fin < 0).sum()),
            "sentinel_code_hits": str(code_hits) if code_hits else "None",
        })
total_pos_inf = sum(int(np.isposinf(pd.to_numeric(soil_df[c], errors="coerce")).sum()) for c in numeric_candidate_cols)
total_neg_inf = sum(int(np.isneginf(pd.to_numeric(soil_df[c], errors="coerce")).sum()) for c in numeric_candidate_cols)
total_negative = sum(int((pd.to_numeric(soil_df[c], errors="coerce").dropna() < 0).sum()) for c in numeric_candidate_cols)
print(f"Across all {len(numeric_candidate_cols)} numeric/numeric-string columns:")
print(f"  Total +inf values    : {total_pos_inf}")
print(f"  Total -inf values    : {total_neg_inf}")
print(f"  Total negative values: {total_negative}")
if inf_audit_rows:
    print(pd.DataFrame(inf_audit_rows).to_string(index=False))
else:
    print("  Zero +inf, zero -inf, zero negative values, and zero -9999/-999/-99/-1/999/9999 codes found.")

In [ ]:
# 5) Categorical consistency, date parsing, and physical/scientific plausibility checks
print("=== 5) CATEGORICAL VALUE AUDIT ===")
for col in ["LabID", "Texture", "Comments"]:
    vc = soil_df[col].value_counts(dropna=False)
    print(f"\n{col} (unique non-null={soil_df[col].nunique(dropna=True)}, missing={int(soil_df[col].isna().sum())}):")
    print(vc.to_string())

print("\n=== 6) DATE PARSING & YEAR CONSISTENCY AUDIT ===")
def classify_date_fmt(val: str) -> str:
    s = val.strip()
    if re.fullmatch(r"\d{1,2}/\d{1,2}/\d{2}", s):
        return "M/D/YY"
    if re.fullmatch(r"\d{1,2}/\d{1,2}/\d{4}", s):
        return "M/D/YYYY"
    if re.fullmatch(r"\d{4}-\d{2}-\d{2}", s):
        return "YYYY-MM-DD"
    return f"OTHER({s})"

parsed_soil_dates = {}
for col in ["Date Received", "Date Reported"]:
    raw = soil_df[col].dropna().astype(str).str.strip()
    fmt_dist = raw.map(classify_date_fmt).value_counts().to_dict()
    dt = pd.to_datetime(soil_df[col], format="mixed", errors="coerce")
    parsed_soil_dates[col] = dt
    n_unparseable = int((soil_df[col].notna() & dt.isna()).sum())
    yr_diff = dt.dt.year - soil_df["Year"]
    n_yr_mismatch = int((dt.notna() & (yr_diff != 0)).sum())
    month_counts = dt.dropna().dt.month.value_counts().sort_index().to_dict()
    print(f"\n{col}: non_null={len(raw)}, parsed={int(dt.notna().sum())}, unparseable={n_unparseable}")
    print(f"  Format distribution : {fmt_dist}")
    print(f"  Month distribution  : {month_counts}")
    print(f"  Year != Trial Year  : {n_yr_mismatch}")
    if n_yr_mismatch > 0:
        mm = soil_df.loc[dt.notna() & (yr_diff != 0), ["Year", "Env", "LabID", col]].copy()
        mm["parsed"] = dt.loc[mm.index].dt.strftime("%Y-%m-%d")
        print(mm.to_string(index=False))

# Diagnostic lab turnaround (Date Reported - Date Received)
rec_dt = parsed_soil_dates["Date Received"]
rep_dt = parsed_soil_dates["Date Reported"]
turnaround_days = (rep_dt - rec_dt).dt.days.dropna()
print(f"\nLab turnaround (Reported - Received, n={len(turnaround_days)}): min={turnaround_days.min():.0f}d, median={turnaround_days.median():.0f}d, max={turnaround_days.max():.0f}d, negative={int((turnaround_days < 0).sum())}")

# 7) Physical / scientific plausibility checks (diagnostic only — no edits)
print("\n=== 7) PHYSICAL / SCIENTIFIC PLAUSIBILITY CHECKS ===")
ph_s = pd.to_numeric(soil_df["1:1 Soil pH"], errors="coerce").dropna()
print(f"1:1 Soil pH outside [0, 14] (impossible): {int(((ph_s < 0) | (ph_s > 14)).sum())} | outside [4.0, 9.0]: {int(((ph_s < 4.0) | (ph_s > 9.0)).sum())}")

ssc = soil_df[["% Sand", "% Silt", "% Clay"]].dropna()
ssc_sum = ssc.sum(axis=1)
print(f"Sand+Silt+Clay sum (n={len(ssc)}): min={ssc_sum.min():.2f}, max={ssc_sum.max():.2f}, rows != 100: {int((ssc_sum.round(2) != 100).sum())}")
if (ssc_sum.round(2) != 100).any():
    bad_ssc = soil_df.loc[ssc_sum[ssc_sum.round(2) != 100].index, ["Year", "Env", "% Sand", "% Silt", "% Clay"]].copy()
    bad_ssc["sum"] = ssc_sum.loc[bad_ssc.index]
    print(bad_ssc.to_string(index=False))

sat_cols = ["%H Sat", "%K Sat", "%Ca Sat", "%Mg Sat", "%Na Sat"]
sat_num = pd.DataFrame({c: pd.to_numeric(soil_df[c], errors="coerce") for c in sat_cols}).dropna()
sat_sum = sat_num.sum(axis=1)
print(f"Cation saturation sum (n={len(sat_num)} complete numeric rows): min={sat_sum.min():.2f}, max={sat_sum.max():.2f}, outside [98, 102]: {int(((sat_sum < 98) | (sat_sum > 102)).sum())}")
if ((sat_sum < 98) | (sat_sum > 102)).any():
    bad_sat = soil_df.loc[sat_sum[(sat_sum < 98) | (sat_sum > 102)].index, ["Year", "Env", "LabID", *sat_cols]].copy()
    bad_sat["sat_sum"] = sat_sum.loc[bad_sat.index].round(2)
    print(bad_sat.to_string(index=False))

# Check percentages outside [0, 100]
pct_cols = ["Organic Matter LOI %", "% Sand", "% Silt", "% Clay", *sat_cols]
for col in pct_cols:
    s = pd.to_numeric(soil_df[col], errors="coerce").dropna()
    bad_pct = int(((s < 0) | (s > 100)).sum())
    if bad_pct > 0:
        print(f"  IMPOSSIBLE percentage outside [0,100] in {col}: {bad_pct}")
print("Percentage bounds [0, 100] audit complete.")

In [ ]:
# Missingness by Year, by LabID, and by Environment
data_cols = [c for c in soil_df.columns if c not in ["Year", "Env", "LabID", "Date Received", "Date Reported", "Comments"]]
soil_df_miss_per_env = soil_df[data_cols].isna().mean(axis=1) * 100

miss_by_year = (
    soil_df.assign(_row_miss_pct=soil_df_miss_per_env)
    .groupby("Year")
    .agg(
        n_envs=("Env", "nunique"),
        mean_col_missing_pct=("_row_miss_pct", "mean"),
        min_col_missing_pct=("_row_miss_pct", "min"),
        max_col_missing_pct=("_row_miss_pct", "max"),
    )
    .round(2)
)
print("=== MISSINGNESS ACROSS 30 SOIL MEASUREMENT COLUMNS BY YEAR ===")
print(miss_by_year.to_string())

print("\n=== SAMPLE COUNTS BY YEAR x LabID ===")
print(pd.crosstab(soil_df["Year"], soil_df["LabID"]).to_string())

print("\n=== MEAN MISSINGNESS (%) ACROSS 30 SOIL MEASUREMENTS BY LabID ===")
print(soil_df.assign(_row_miss_pct=soil_df_miss_per_env).groupby("LabID")["_row_miss_pct"].agg(["count", "mean", "min", "max"]).round(2).to_string())

env_miss_summary = soil_df[["Year", "Env", "LabID"]].assign(missing_pct=soil_df_miss_per_env.round(2))
print("\nTop 10 environments by soil-measurement missingness (%):")
print(env_miss_summary.sort_values("missing_pct", ascending=False).head(10).to_string(index=False))

# Local missingness plot
plot_miss = schema_df[schema_df["missing_count"] > 0].sort_values("missing_pct", ascending=True)
fig, ax = plt.subplots(figsize=(10.5, 9.5))
bars = ax.barh(plot_miss["column"], plot_miss["missing_pct"], color="#2b6cb0", edgecolor="white")
ax.set_xlabel("Missing Values (%)")
ax.set_title(f"Missing Value Percentage by Column — {SOIL_PATH.name} (n = {len(soil_df):,})")
ax.set_xlim(0, 118)
for bar, (_, row) in zip(bars, plot_miss.iterrows()):
    ax.text(
        bar.get_width() + 0.8,
        bar.get_y() + bar.get_height() / 2,
        f"{row['missing_pct']:.1f}% ({int(row['missing_count'])})",
        va="center",
        fontsize=8,
    )
plt.tight_layout()
fig_path_1 = LOCAL_FIGURES_DIR / "05_soil_missing_values.png"
fig.savefig(fig_path_1, dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)
print(f"Saved local figure: {fig_path_1.relative_to(PROJECT_ROOT).as_posix()}")

## 4. Numerical Distributions & Outlier Diagnostics

### Statistical Outliers vs. Scientifically Impossible Measurements
The $1.5 \times \text{IQR}$ rule flags statistical tail values relative to the middle 50% of the data. In soil science, skewed distributions (e.g., high `Nitrate-N`, elevated `Mehlich P-III`, high `Organic Matter` in peat/muck soils, or elevated soluble salts) naturally produce many statistical IQR outliers that are **scientifically real and agronomically meaningful**. We quantify IQR outliers explicitly without removing or altering any observations, separating them clearly from scientifically impossible values.

In [ ]:
# Compute full numerical distribution summary on finite numerical observations
# Includes numeric columns + numerically coerced finite values of %H Sat and %Na Sat (excluding censored '< 0.01' strings)
exclude_from_dist = {"Year", "Env", "LabID", "Date Received", "Date Reported", "Texture", "Comments"}
num_soil_cols = [c for c in soil_df.columns if c not in exclude_from_dist]

dist_rows = []
for col in num_soil_cols:
    coerced = pd.to_numeric(soil_df[col], errors="coerce")
    fin = coerced[np.isfinite(coerced)]
    n_fin = int(len(fin))
    if n_fin == 0:
        continue
    p01, p05, q25, q50, q75, p95, p99 = np.percentile(fin, [1, 5, 25, 50, 75, 95, 99])
    iqr = float(q75 - q25)
    lo_fence = q25 - 1.5 * iqr
    hi_fence = q75 + 1.5 * iqr
    n_lo = int((fin < lo_fence).sum())
    n_hi = int((fin > hi_fence).sum())
    n_out = n_lo + n_hi
    dist_rows.append({
        "column": col,
        "n_fin": n_fin,
        "min": round(float(fin.min()), 3),
        "p05": round(float(p05), 3),
        "q1": round(float(q25), 3),
        "median": round(float(q50), 3),
        "mean": round(float(fin.mean()), 3),
        "q3": round(float(q75), 3),
        "p95": round(float(p95), 3),
        "max": round(float(fin.max()), 3),
        "std": round(float(fin.std(ddof=1)), 3) if n_fin > 1 else np.nan,
        "iqr": round(iqr, 3),
        "n_zero": int((fin == 0).sum()),
        "iqr_out": n_out,
        "iqr_pct": round(n_out / n_fin * 100, 1),
    })

dist_df = pd.DataFrame(dist_rows)
print("=== NUMERICAL SOIL DISTRIBUTIONS (FINITE OBSERVATIONS; NO OUTLIERS REMOVED) ===")
print(dist_df.to_string(index=False))

# Also report P01 and P99 for all numeric soil columns
p_extreme_rows = []
for col in num_soil_cols:
    fin = pd.to_numeric(soil_df[col], errors="coerce").dropna()
    fin = fin[np.isfinite(fin)]
    if len(fin):
        p01, p99 = np.percentile(fin, [1, 99])
        p_extreme_rows.append({"column": col, "n_fin": len(fin), "p01": round(float(p01), 3), "p99": round(float(p99), 3)})
print("\n=== 1st AND 99th PERCENTILES ===")
print(pd.DataFrame(p_extreme_rows).to_string(index=False))

In [ ]:
# Local visualizations: Core well-populated numerical soil distributions
core_plot_cols = [
    "1:1 Soil pH",
    "Organic Matter LOI %",
    "CEC/Sum of Cations me/100g",
    "1:1 S Salts mmho/cm",
    "Nitrate-N ppm N",
    "Mehlich P-III ppm P",
    "Potassium ppm K",
    "% Sand",
    "% Clay",
]
fig, axes = plt.subplots(3, 3, figsize=(15, 11))
for ax, col in zip(axes.flat, core_plot_cols):
    s = pd.to_numeric(soil_df[col], errors="coerce").dropna()
    s = s[np.isfinite(s)]
    ax.hist(s, bins=25, color="#2f855a", alpha=0.82, edgecolor="white")
    ax.axvline(s.median(), color="#1a365d", linestyle="-", linewidth=1.4, label=f"Med={s.median():.2f}")
    ax.axvline(s.mean(), color="#c53030", linestyle="--", linewidth=1.3, label=f"Mean={s.mean():.2f}")
    ax.set_title(f"{col} (n={len(s)})", fontsize=10)
    ax.set_ylabel("Envs")
    ax.legend(fontsize=7.5, frameon=True)

plt.suptitle(f"Core Numerical Soil Property Distributions — {SOIL_PATH.name}", y=1.01, fontsize=12)
plt.tight_layout()
fig_path_2 = LOCAL_FIGURES_DIR / "06_soil_core_distributions.png"
fig.savefig(fig_path_2, dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)
print(f"Saved local figure: {fig_path_2.relative_to(PROJECT_ROOT).as_posix()}")

# Texture classes & LabID distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
tex_counts = soil_df["Texture"].dropna().value_counts().sort_values(ascending=True)
axes[0].barh(tex_counts.index, tex_counts.values, color="#805ad5", edgecolor="white")
axes[0].set_title(f"USDA Soil Texture Classes (n = {int(soil_df['Texture'].notna().sum())} envs)")
axes[0].set_xlabel("Environment Count")

lab_by_yr = pd.crosstab(soil_df["Year"], soil_df["LabID"])
lab_by_yr.plot(kind="bar", stacked=True, ax=axes[1], colormap="Set2", edgecolor="white")
axes[1].set_title(f"Soil Samples by Year and Laboratory ({int(soil_df['Year'].min())}–{int(soil_df['Year'].max())})")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Environment Count")
axes[1].tick_params(axis="x", rotation=45)
axes[1].legend(title="LabID", fontsize=8, title_fontsize=8.5)

plt.tight_layout()
fig_path_3 = LOCAL_FIGURES_DIR / "07_soil_texture_and_labs.png"
fig.savefig(fig_path_3, dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)
print(f"Saved local figure: {fig_path_3.relative_to(PROJECT_ROOT).as_posix()}")

## 5. Observation Granularity & Multi-Year Soil Variability

Here we verify whether `Env` uniquely identifies every soil record, inspect sampling depth (`E Depth`) and sampling/reporting date distributions, and diagnose how much key soil variables vary across years within the same location prefix (distinguishing relatively stable properties like soil texture/pH/OM from mobile seasonal nutrients like nitrate-N). No aggregation or modification is performed.

In [ ]:
env_is_unique = bool(soil_df["Env"].is_unique)
n_unique_env = int(soil_df["Env"].nunique(dropna=True))
dup_env_rows = int(soil_df.duplicated(subset=["Env"], keep=False).sum())

env_yr_suffix = soil_df["Env"].str.extract(r"_(\d{4})$")[0]
env_yr_mismatch = int((env_yr_suffix.astype("Int64") != soil_df["Year"]).sum())

print("=== SOIL OBSERVATION GRANULARITY ===")
print(f"Total soil rows                  : {len(soil_df):,}")
print(f"Unique Env identifiers           : {n_unique_env:,}")
print(f"Env uniquely identifies each row : {env_is_unique}")
print(f"Rows with duplicate Env          : {dup_env_rows:,}")
print(f"Env _YYYY suffix != Year rows    : {env_yr_mismatch}")
print(f"Years covered                    : {int(soil_df['Year'].min())}–{int(soil_df['Year'].max())} ({soil_df['Year'].nunique()} unique years)")
print(f"E Depth value counts             : {soil_df['E Depth'].value_counts(dropna=False).to_dict()}")
print(f"Texture No value counts          : {soil_df['Texture No'].value_counts(dropna=False).to_dict()}")

# Diagnostic cross-year variability within recurring Env location prefixes (read-only)
soil_site = soil_df.assign(_site_prefix=soil_df["Env"].str.replace(r"_\d{4}$", "", regex=True))
diag_vars = ["1:1 Soil pH", "Organic Matter LOI %", "CEC/Sum of Cations me/100g", "% Sand", "% Clay", "Nitrate-N ppm N", "Mehlich P-III ppm P", "Potassium ppm K"]

var_rows = []
for col in diag_vars:
    tmp = soil_site[["_site_prefix", col]].dropna()
    grp = tmp.groupby("_site_prefix")[col].agg(["count", "mean", "std"]).query("count >= 3")
    global_std = float(tmp[col].std(ddof=1))
    var_rows.append({
        "variable": col,
        "sites_ge3_yrs": len(grp),
        "global_std": round(global_std, 3),
        "median_within_site_std": round(float(grp["std"].median()), 3) if len(grp) else np.nan,
        "within_to_global_ratio": round(float(grp["std"].median()) / global_std, 3) if (len(grp) and global_std > 0) else np.nan,
    })
print("\n=== DIAGNOSTIC WITHIN-SITE CROSS-YEAR VARIABILITY (SITES WITH >= 3 YEARS) ===")
print(pd.DataFrame(var_rows).to_string(index=False))

## 6. Cross-Dataset Coverage Audit (Trait vs. Metadata vs. Soil — Read-Only)

We load strictly the required identifier and target columns (`Env`, `Year`, `Yield_Mg_ha` from trait; `Env`, `Year` from metadata) to audit environment coverage across all three datasets by year, quantify affected trait and valid-yield observations, verify the absence of 2014 soil data and incomplete coverage in other years, and confirm join cardinality — without creating or persisting any merged dataset.

In [ ]:
trait_keys = pd.read_csv(TRAIT_PATH, engine="c", usecols=["Env", "Year", "Yield_Mg_ha"], low_memory=False)
meta_keys = pd.read_csv(META_PATH, engine="c", usecols=["Env", "Year"], low_memory=False)
trait_finite_yield = np.isfinite(pd.to_numeric(trait_keys["Yield_Mg_ha"], errors="coerce"))

trait_envs = set(trait_keys["Env"].dropna())
meta_envs = set(meta_keys["Env"].dropna())
soil_envs = set(soil_df["Env"].dropna())

trait_with_soil = trait_envs & soil_envs
meta_with_soil = meta_envs & soil_envs
all_three = trait_envs & meta_envs & soil_envs

trait_no_soil = sorted(trait_envs - soil_envs)
soil_no_trait = sorted(soil_envs - trait_envs)
soil_no_meta = sorted(soil_envs - meta_envs)

no_soil_mask = ~trait_keys["Env"].isin(soil_envs)
n_trait_no_soil = int(no_soil_mask.sum())
n_valid_yield_no_soil = int((no_soil_mask & trait_finite_yield).sum())

print("=== CROSS-DATASET Env COVERAGE SUMMARY (READ-ONLY) ===")
print(f"Unique Env — Trait / Metadata / Soil          : {len(trait_envs):,} / {len(meta_envs):,} / {len(soil_envs):,}")
print(f"Matching Env — Trait & Soil                   : {len(trait_with_soil):,} ({len(trait_with_soil) / len(trait_envs) * 100:.2f}% of trait envs)")
print(f"Matching Env — Metadata & Soil                : {len(meta_with_soil):,} ({len(meta_with_soil) / len(meta_envs) * 100:.2f}% of meta envs)")
print(f"Matching Env — All three (Trait & Meta & Soil): {len(all_three):,}")
print(f"Trait Envs WITHOUT Soil data                  : {len(trait_no_soil):,}")
print(f"Soil Envs missing from Trait                  : {len(soil_no_trait):,} -> {soil_no_trait}")
print(f"Soil Envs missing from Metadata               : {len(soil_no_meta):,} -> {soil_no_meta}")
print(f"Trait obs affected (no soil data)             : {n_trait_no_soil:,} / {len(trait_keys):,} ({n_trait_no_soil / len(trait_keys) * 100:.2f}%)")
print(f"Valid finite yield obs affected (no soil)     : {n_valid_yield_no_soil:,} / {int(trait_finite_yield.sum()):,} ({n_valid_yield_no_soil / int(trait_finite_yield.sum()) * 100:.2f}%)")
print(f"Expected join cardinality (Trait -> Soil)     : many-to-one (soil Env unique={soil_df['Env'].is_unique})")
print(f"Expected join cardinality (Meta <-> Soil)     : one-to-one on matched Envs (both Env unique={meta_keys['Env'].is_unique and soil_df['Env'].is_unique})")

# Year-by-year coverage breakdown (2014-2025)
all_years = sorted(set(trait_keys["Year"].unique()) | set(meta_keys["Year"].unique()) | set(soil_df["Year"].unique()))
yr_cov_rows = []
for yr in all_years:
    t_sub = trait_keys[trait_keys["Year"] == yr]
    t_fin = trait_finite_yield[trait_keys["Year"] == yr]
    t_envs_y = set(t_sub["Env"].dropna())
    m_envs_y = set(meta_keys.loc[meta_keys["Year"] == yr, "Env"].dropna())
    s_envs_y = set(soil_df.loc[soil_df["Year"] == yr, "Env"].dropna())
    matched_y = t_envs_y & s_envs_y
    no_soil_y = ~t_sub["Env"].isin(soil_envs)
    yr_cov_rows.append({
        "Year": yr,
        "trait_envs": len(t_envs_y),
        "meta_envs": len(m_envs_y),
        "soil_envs": len(s_envs_y),
        "matched_trait_soil_envs": len(matched_y),
        "env_cov_pct": round(len(matched_y) / len(t_envs_y) * 100, 1) if len(t_envs_y) else 0.0,
        "trait_obs_no_soil": int(no_soil_y.sum()),
        "valid_yield_no_soil": int((no_soil_y & t_fin).sum()),
        "valid_yield_cov_pct": round((1 - (no_soil_y & t_fin).sum() / t_fin.sum()) * 100, 1) if t_fin.sum() else 0.0,
    })

yr_cov_df = pd.DataFrame(yr_cov_rows)
print("\n=== SOIL COVERAGE BY YEAR (2014-2025) ===")
print(yr_cov_df.to_string(index=False))

# Check whether any trait environments missing from metadata appear in soil data
tx_in_soil = [e for e in sorted(trait_envs - meta_envs) if e in soil_envs]
print(f"\nTrait Envs missing from Metadata ({sorted(trait_envs - meta_envs)}) present in Soil: {tx_in_soil}")

In [ ]:
# Local visualization: Soil coverage by year vs Trait & Metadata
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
yrs = yr_cov_df["Year"].values
x = np.arange(len(yrs))
w = 0.28

axes[0].bar(x - w, yr_cov_df["trait_envs"], width=w, color="#3182ce", label="Trait Envs")
axes[0].bar(x, yr_cov_df["meta_envs"], width=w, color="#2f855a", label="Metadata Envs")
axes[0].bar(x + w, yr_cov_df["soil_envs"], width=w, color="#dd6b20", label="Soil Envs")
axes[0].set_title(f"Environment Coverage by Year ({int(yrs.min())}–{int(yrs.max())})")
axes[0].set_xticks(x)
axes[0].set_xticklabels(yrs, rotation=45)
axes[0].set_ylabel("Unique Environments")
axes[0].legend(frameon=True, fontsize=8.5)

axes[1].bar(yrs, yr_cov_df["valid_yield_cov_pct"], color="#805ad5", edgecolor="white")
axes[1].set_title(f"Valid Yield Plots Covered by Soil Data ({int(yrs.min())}–{int(yrs.max())})")
axes[1].set_xlabel("Growing Season Year")
axes[1].set_ylabel("Valid Yield Observations Covered (%)")
axes[1].set_xticks(yrs)
axes[1].tick_params(axis="x", rotation=45)
axes[1].set_ylim(0, 112)
for yr_v, pct_v in zip(yrs, yr_cov_df["valid_yield_cov_pct"]):
    axes[1].text(yr_v, pct_v + 1.8, f"{pct_v:.0f}%", ha="center", fontsize=8)

plt.tight_layout()
fig_path_4 = LOCAL_FIGURES_DIR / "08_soil_coverage_by_year.png"
fig.savefig(fig_path_4, dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)
print(f"Saved local figure: {fig_path_4.relative_to(PROJECT_ROOT).as_posix()}")

## 7. Preliminary Feature Eligibility & Leakage Assessment

### Guardrail
Soil measurements are generally collected pre-season or early-season to characterize baseline field fertility — **however**, auditing `Date Received`, `Date Reported`, and `Comments` shows that some soil samples were received late in the season or explicitly noted as sampled post-season/out-of-season. Furthermore, final eligibility for EVERY soil column remains strictly subject to:
1. Verification against the **2026 testing soil schema and coverage**.
2. Confirmation of **measurement timing** (especially for mobile nutrients such as `Nitrate-N ppm N` / `lbs N/A` and lab reporting timestamps).
3. Official competition rules.

In [ ]:
soil_eligibility = [
    # Identifiers
    ("Env", "Identifiers", "Primary environment join key across G2F tables and CV grouping key."),
    ("Year", "Identifiers", "Trial growing season year; critical for temporal CV splitting."),
    # Potential temporal or sampling-context features
    ("LabID", "Potential temporal or sampling-context features", "Testing laboratory identifier; useful for diagnosing batch/method differences across labs."),
    ("Date Received", "Potential temporal or sampling-context features", "Lab receipt date; diagnostic for pre-season vs. in-season/post-season sampling timing."),
    ("E Depth", "Potential temporal or sampling-context features", "Soil core sampling depth; verify unit (likely inches) and 2026 test availability."),
    # Potential leakage risks (provisionally excluded pending 2026 test-schema & timing verification)
    ("Date Reported", "Potential leakage risks", "Lab report date (often post-planting or post-season); non-causal administrative timestamp, provisionally excluded."),
    ("Comments", "Potential leakage risks", "Free-text lab/sampling notes (includes post-season/out-of-season notes); provisionally excluded pending test-schema audit."),
    # Potential categorical soil features
    ("Texture", "Potential categorical soil features", "USDA soil texture class; normalize hyphenated/rare classes and verify 2026 test availability."),
    # Potential numerical soil features
    ("1:1 Soil pH", "Potential numerical soil features", "Core soil acidity/alkalinity measure; verify 2026 test availability."),
    ("WDRF Buffer pH", "Potential numerical soil features", "Buffer pH (reserve acidity) from primary lab panel; verify 2026 availability."),
    ("1:1 S Salts mmho/cm", "Potential numerical soil features", "Soluble salts / electrical conductivity; verify 2026 availability."),
    ("Organic Matter LOI %", "Potential numerical soil features", "Soil organic matter by loss-on-ignition; core fertility/water-retention feature pending 2026 audit."),
    ("Nitrate-N ppm N", "Potential numerical soil features", "Mobile plant-available nitrate-N; check sampling timing sensitivity (pre- vs. post-season) and 2026 availability."),
    ("lbs N/A", "Potential numerical soil features", "Estimated available nitrogen (lbs N/acre); collinear with Nitrate-N/depth; verify timing and 2026 availability."),
    ("Potassium ppm K", "Potential numerical soil features", "Extractable soil potassium; verify cross-lab method comparability and 2026 availability."),
    ("Sulfate-S ppm S", "Potential numerical soil features", "Extractable sulfate sulfur; verify 2026 availability."),
    ("Calcium ppm Ca", "Potential numerical soil features", "Exchangeable calcium; verify 2026 availability."),
    ("Magnesium ppm Mg", "Potential numerical soil features", "Exchangeable magnesium; verify 2026 availability."),
    ("Sodium ppm Na", "Potential numerical soil features", "Exchangeable sodium; verify 2026 availability."),
    ("CEC/Sum of Cations me/100g", "Potential numerical soil features", "Cation exchange capacity; strong soil fertility proxy pending 2026 audit."),
    ("%H Sat", "Potential numerical soil features", "Hydrogen saturation %; contains censored '< 0.01' strings requiring careful numeric handling in Phase 3."),
    ("%K Sat", "Potential numerical soil features", "Potassium saturation %; verify 2026 availability."),
    ("%Ca Sat", "Potential numerical soil features", "Calcium saturation %; verify 2026 availability."),
    ("%Mg Sat", "Potential numerical soil features", "Magnesium saturation %; verify 2026 availability."),
    ("%Na Sat", "Potential numerical soil features", "Sodium saturation %; contains censored '< 0.01' string requiring careful numeric handling in Phase 3."),
    ("Mehlich P-III ppm P", "Potential numerical soil features", "Extractable soil phosphorus (Mehlich-3); verify cross-lab comparability and 2026 availability."),
    ("% Sand", "Potential numerical soil features", "Sand particle fraction (%); relatively stable site property pending 2026 audit."),
    ("% Silt", "Potential numerical soil features", "Silt particle fraction (%); collinear with % Sand and % Clay (sums to ~100%)."),
    ("% Clay", "Potential numerical soil features", "Clay particle fraction (%); controls water retention and CEC pending 2026 audit."),
    # Unknown variables requiring further investigation (undocumented coding or ultra-sparse lab-specific analytes)
    ("Texture No", "Unknown variables requiring further investigation", "Numeric texture code without local data dictionary; verify coding definition before use."),
    ("BpH", "Unknown variables requiring further investigation", "Ultra-sparse alternate buffer pH column (lab-specific); verify relationship to WDRF Buffer pH and 2026 presence."),
    ("Zinc ppm Zn", "Unknown variables requiring further investigation", "Ultra-sparse micronutrient column; verify whether populated in 2026 test soil data."),
    ("Iron ppm Fe", "Unknown variables requiring further investigation", "Ultra-sparse micronutrient column; verify whether populated in 2026 test soil data."),
    ("Manganese ppm Mn", "Unknown variables requiring further investigation", "Ultra-sparse micronutrient column; verify whether populated in 2026 test soil data."),
    ("Copper ppm Cu", "Unknown variables requiring further investigation", "Ultra-sparse micronutrient column; verify whether populated in 2026 test soil data."),
    ("Boron ppm B", "Unknown variables requiring further investigation", "Ultra-sparse micronutrient column; verify whether populated in 2026 test soil data."),
]

elig_df = pd.DataFrame(soil_eligibility, columns=["column", "category", "rationale"])
elig_df["missing_pct"] = [round(float(soil_df[c].isna().mean() * 100), 2) for c in elig_df["column"]]
assert set(elig_df["column"]) == set(soil_df.columns), "All soil columns must be classified!"
print("=== PRELIMINARY SOIL FEATURE ELIGIBILITY CLASSIFICATION ===")
for cat, grp in elig_df.groupby("category", sort=False):
    print(f"  {cat} ({len(grp)}): {list(grp['column'])}")

In [ ]:
# Dynamic end-of-notebook verification summary (zero hardcoded statistics)
print("=== DYNAMIC PHASE 2.4 SUMMARY (COMPUTED LIVE) ===")
print(f"Soil shape                                    : {soil_df.shape[0]:,} rows x {soil_df.shape[1]} columns")
print(f"Load time / Deep memory                       : {load_seconds:.4f} s / {mem_kb_deep:.2f} KB ({mem_mb_deep:.4f} MB)")
print(f"Exact duplicates / Env unique                 : {exact_dups:,} / {env_is_unique}")
print(f"Years covered                                 : {int(soil_df['Year'].min())}-{int(soil_df['Year'].max())} ({soil_df['Year'].nunique()} years)")
print(f"Unique LabID values                           : {soil_df['LabID'].nunique()} -> {soil_df['LabID'].value_counts().to_dict()}")
print(f"String-typed numeric columns w/ non-numeric   : { {k: v['non_numeric_tokens'] for k, v in non_numeric_issues.items()} }")
print(f"Total +inf / -inf / negative numeric values   : {total_pos_inf} / {total_neg_inf} / {total_negative}")
print(f"Sand+Silt+Clay sum != 100 count               : {int((ssc_sum.round(2) != 100).sum())} / {len(ssc)}")
print(f"Cation sat sum outside [98, 102] count        : {int(((sat_sum < 98) | (sat_sum > 102)).sum())} / {len(sat_num)}")
print(f"Trait / Meta / Soil unique Envs               : {len(trait_envs):,} / {len(meta_envs):,} / {len(soil_envs):,}")
print(f"Trait Envs matched / missing soil             : {len(trait_with_soil):,} / {len(trait_no_soil):,}")
print(f"Trait obs / Valid yield obs without soil      : {n_trait_no_soil:,} / {n_valid_yield_no_soil:,}")
print(f"2014 soil environments                        : {int(yr_cov_df.loc[yr_cov_df['Year'] == 2014, 'soil_envs'].iloc[0])}")
print(f"Join cardinality (Trait -> Soil)              : many-to-one (verified={env_is_unique})")

## 8. Confirmed Structural Takeaways & Unresolved Technical Questions

*(All exact numerical findings are computed dynamically above and archived locally in `docs/SOIL_DATA_EDA.md` pending clarification of competition data-sharing permissions.)*

### Confirmed Structural Findings
- **Granularity & Relational Integrity:** `Env` uniquely identifies every soil record with zero exact duplicate rows, confirming a **many-to-one** relationship from plot-level trait records to environment-level soil records (and one-to-one with metadata on matched environments).
- **Incomplete Temporal & Environmental Coverage:** Soil data begins in `2015` (completely absent for `2014`) and has partial missingness across subsequent seasons, requiring strict **left-join** integration and leakage-safe missingness handling in Phase 3+.
- **Laboratory & Detection-Limit Nuances:** Multiple testing laboratories contribute different analyte panels (causing block-structured missingness for alternate buffer pH and micronutrients), and below-detection-limit censored strings (`< 0.01`) coerce `%H Sat` and `%Na Sat` to string dtypes under default CSV loading.
- **Sampling-Timing Awareness:** Collaborator comments and lab receipt/report dates show that a subset of soil samples were collected or processed late/post-season, requiring sampling-timing audits especially for mobile nutrients (e.g., nitrate-N) before modeling.

### Unresolved Technical Questions (For Phase 2.5+ / Phase 3)
1. **2026 Test Soil Schema & Coverage:** Verify which soil variables, laboratories, and environments are present in 2026 testing data before approving any soil feature.
2. **Handling Missing Soil Years (`2014` and Partial Seasons):** Decide in Phase 3 whether to rely on tree-native missing handling + missingness indicators, fold-local imputation, or external public soil maps (respecting the Feb 1, 2026 external-data cutoff).
3. **Censored Below-Detection-Limit Policy:** Establish a reproducible Phase 3 rule for handling `< 0.01` tokens in `%H Sat` and `%Na Sat` (e.g., half detection limit `0.005` vs. `0.0` + indicator).
4. **Undocumented Units & Codes:** Confirm exact units for `E Depth` (likely inches) and the exact numeric coding of `Texture No`, plus cross-lab equivalence between `BpH` and `WDRF Buffer pH` and across laboratory phosphorus/potassium methods.
5. **Sampling-Timing Leakage Audit:** Verify whether post-season/out-of-season soil samples introduce any leakage for mobile nutrients (`Nitrate-N ppm N`, `lbs N/A`).